# 1. Notebook Purpose

This notebook demonstrates the proof-of-concept AI analysis stage of the accessibility scanning system.

It processes scan requests that have already completed a baseline accessibility scan, then applies two additional checks:

1. Alt text relevance analysis
2. Heading meaningfulness analysis

The notebook extracts images, alt text, headings, and related page content from a scanned URL. It then creates structured AI findings that can be submitted back to the project API.

The full project rationale, methodology, limitations, and evaluation are discussed in the main report. This appendix focuses on the practical implementation of the AI analysis workflow.

# 2. Package Installation

The following packages are required to run the notebook.

In [1]:
%pip install requests beautifulsoup4 pillow transformers torch sentence-transformers scikit-learn pandas

Defaulting to user installation because normal site-packages is not writeableNote: you may need to restart the kernel to use updated packages.




[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: C:\Users\cfowler\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In a production version, these dependencies would be managed using a dedicated environment file rather than being installed directly in the notebook.

# 3. Imports

This section imports the Python libraries used throughout the notebook.

In [2]:
import io
import re
from dataclasses import dataclass
from typing import Optional
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup
from PIL import Image

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import torch

from sentence_transformers import SentenceTransformer, util
from transformers import BlipForConditionalGeneration, BlipProcessor

C:\Users\cfowler\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 4. Data Structures

This section defines simple data structures used to store images and headings extracted from a webpage.

In [3]:
@dataclass
class PageImage:
    src: str
    alt_text: Optional[str]


@dataclass
class PageHeading:
    level: int
    text: str
    following_text: str

`PageImage` stores the image URL and its alt text. The alt text is optional because some images may not have an `alt` attribute.

`PageHeading` stores the heading level, the heading text, and the page content that follows the heading.

# 5. API Configuration

This section stores the base URL for the local API and defines a timeout value for HTTP requests.

In [4]:
API_BASE_URL = "https://localhost:7290"

REQUEST_TIMEOUT_SECONDS = 30

The API base URL points to the local .NET API used by the project. The timeout value prevents requests from hanging indefinitely if the API or target webpage does not respond.

# 6. API Helper Functions

This section defines small helper functions for communicating with the project API.

The notebook uses these functions to:

- Get scan requests waiting for AI analysis.
- Submit AI findings back to the API.
- Mark an AI scan as complete.

In [5]:
def get_pending_ai_scans() -> list[dict]:
    response = requests.get(
        f"{API_BASE_URL}/api/scans/pending-ai",
        timeout=REQUEST_TIMEOUT_SECONDS,
        verify=False
    )

    response.raise_for_status()

    return response.json()


def post_ai_finding(scan_id: int, finding: dict) -> None:
    response = requests.post(
        f"{API_BASE_URL}/api/scans/{scan_id}/ai-findings",
        json=finding,
        timeout=REQUEST_TIMEOUT_SECONDS,
        verify=False
    )

    response.raise_for_status()


def mark_ai_scan_complete(scan_id: int) -> None:
    response = requests.post(
        f"{API_BASE_URL}/api/scans/{scan_id}/ai-complete",
        timeout=REQUEST_TIMEOUT_SECONDS,
        verify=False
    )

    response.raise_for_status()

`verify=False` is used because the API is running locally over HTTPS with a development certificate. This would not be suitable for a production system.

# 7. Get Pending AI Scan Requests

This section retrieves scan requests that are waiting for AI analysis.

For this proof of concept, the notebook processes one scan request per run. This keeps the workflow easier to test, inspect, and explain.

In [312]:
pending_scans = get_pending_ai_scans()

if len(pending_scans) == 0:
    raise ValueError("No scans are currently waiting for AI analysis.")

print(f"Pending AI scans found: {len(pending_scans)}")

Pending AI scans found: 1


C:\Users\cfowler\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'localhost'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [313]:
pending_scans_dataframe = pd.DataFrame(pending_scans)

pending_scans_dataframe

,id,url,dateTimeCreated,baselineScanIsCompleted,baselineScanDateTimeCompleted,aiScanIsCompleted,aiScanDateTimeCompleted
0,31,http://localhost:8000/mock_pages/05_severe_com...,2026-06-13T21:28:59.7891877,True,2026-06-13T21:29:02.0846636,False,None


The notebook will process each pending scan request in turn. This allows multiple URLs to be analysed without manually selecting one scan at a time.

In [314]:
selected_scan = pending_scans[0]

scan_id = selected_scan["id"]
scan_url = selected_scan["url"]

print(f"Selected scan ID: {scan_id}")
print(f"Selected scan URL: {scan_url}")

Selected scan ID: 31
Selected scan URL: http://localhost:8000/mock_pages/05_severe_combined_issues.html


Only the first pending scan is selected in this notebook. A future improvement would be to process all pending scans automatically.

# 8. Fetch and Parse the Webpage

This section downloads the HTML for the selected scan URL and parses it using BeautifulSoup.

In [315]:
def get_page_html(url: str) -> str:
    response = requests.get(
        url,
        timeout=REQUEST_TIMEOUT_SECONDS
    )

    response.raise_for_status()

    return response.text

In [316]:
page_html = get_page_html(scan_url)

print(f"Downloaded {len(page_html)} characters from {scan_url}")

Downloaded 1544 characters from http://localhost:8000/mock_pages/05_severe_combined_issues.html


In [317]:
soup = BeautifulSoup(page_html, "html.parser")

page_title = soup.title.get_text(strip=True) if soup.title is not None else "No page title found"

print(f"Page title: {page_title}")

Page title: Mock Accessibility Page 04 - Review decorative and context dependent


The parsed HTML is stored in the `soup` variable. This will be used in later sections to extract images, alt text, headings, and related page content.

# 9. Extract Images from the Page

This section extracts all image elements from the parsed webpage.

For each image, the notebook stores:

- The image source URL
- The alt text value
- Whether the alt attribute is present

In [318]:
def get_absolute_url(page_url: str, src: str) -> str:
    return urljoin(page_url, src)


def extract_images(soup: BeautifulSoup, page_url: str) -> list[PageImage]:
    images: list[PageImage] = []

    for image in soup.find_all("img"):
        src = image.get("src")

        if src is None:
            continue

        absolute_src = get_absolute_url(page_url, src)
        alt_text = image.get("alt")

        images.append(
            PageImage(
                src=absolute_src,
                alt_text=alt_text
            )
        )

    return images

In [319]:
page_images = extract_images(soup, scan_url)

print(f"Images found: {len(page_images)}")

Images found: 3


In [320]:
images_dataframe = pd.DataFrame(
    [
        {
            "src": image.src,
            "alt_text": image.alt_text,
            "has_alt_attribute": image.alt_text is not None
        }
        for image in page_images
    ]
)

images_dataframe

,src,alt_text,has_alt_attribute
0,https://www.aberdeenshire.gov.uk/media/hqrc0hy...,,True
1,https://www.aberdeenshire.gov.uk/media/rrtfeee...,,True
2,https://www.aberdeenshire.gov.uk/media/17236/2...,,True


The `has_alt_attribute` column helps distinguish between an image with no alt attribute and an image with an empty alt attribute. This distinction is important because decorative images may correctly use empty alt text.

# 10. Extract Headings and Related Content

This section extracts headings from the webpage and collects the content that follows each heading.

The following content is used later to estimate whether a heading is meaningful and relevant.

In [349]:
def get_heading_level(heading) -> int:
    return int(heading.name[1])


def extract_headings(soup: BeautifulSoup) -> list[PageHeading]:
    headings: list[PageHeading] = []

    page_root = soup.find("main")

    if page_root is None:
        page_root = soup.body

    if page_root is None:
        page_root = soup

    heading_tags = ["h1", "h2", "h3", "h4", "h5", "h6"]

    for heading in page_root.find_all(heading_tags):
        heading_text = heading.get_text(" ", strip=True)

        if heading_text == "":
            continue

        heading_level = get_heading_level(heading)

        following_parts: list[str] = []

        for element in heading.find_all_next():
            if element.name in heading_tags:
                next_heading_level = get_heading_level(element)

                if next_heading_level <= heading_level:
                    break

            if element.name in ["p", "li"]:
                text = element.get_text(" ", strip=True)

                if text != "":
                    following_parts.append(text)

        following_text = " ".join(following_parts)

        headings.append(
            PageHeading(
                level=heading_level,
                text=heading_text,
                following_text=following_text
            )
        )

    return headings

In [322]:
page_headings = extract_headings(soup)

print(f"Headings found: {len(page_headings)}")

Headings found: 5


In [323]:
headings_dataframe = pd.DataFrame(
    [
        {
            "level": heading.level,
            "text": heading.text,
            "following_text": heading.following_text
        }
        for heading in page_headings
    ]
)

headings_dataframe

,level,text,following_text
0,1,Housing and community buildings,This mock page demonstrates that empty alt tex...
1,2,Affordable housing development,New affordable housing can provide modern rent...
2,2,New council building progress,Community buildings can bring several services...
3,2,Decorative separator image,The next image is being used as a decorative v...
4,2,Review required,The AI tool should not automatically fail all ...


In [324]:
headings_preview_dataframe = pd.DataFrame(
    [
        {
            "level": heading.level,
            "text": heading.text,
            "following_text_preview": heading.following_text[:250]
        }
        for heading in page_headings
    ]
)

headings_preview_dataframe

,level,text,following_text_preview
0,1,Housing and community buildings,This mock page demonstrates that empty alt tex...
1,2,Affordable housing development,New affordable housing can provide modern rent...
2,2,New council building progress,Community buildings can bring several services...
3,2,Decorative separator image,The next image is being used as a decorative v...
4,2,Review required,The AI tool should not automatically fail all ...


# 11. Alt Text Rule-Based Checks

This section applies simple rule-based checks to the alt text found on each image.

These checks identify common issues such as:

- Missing alt attributes
- Generic alt text
- Very short alt text
- File names being used as alt text

Empty alt text is not automatically treated as a failure because some decorative images should use `alt=""`.

In [ ]:
GENERIC_ALT_TEXT_VALUES = {
    "image",
    "photo",
    "picture",
    "graphic",
    "icon",
    "logo",
    "img",
    "banner",
    "placeholder",
    "button",
    "link",
    "thumbnail",
    "untitled",
    "click here",
    "read more"
}


def clean_text(value: str) -> str:
    value = value.lower()
    value = re.sub(r"[^a-z0-9\s]", " ", value)
    value = re.sub(r"\s+", " ", value)

    return value.strip()


def looks_like_filename(value: str) -> bool:
    cleaned_value = value.lower().strip()

    image_extensions = [
        ".jpg",
        ".jpeg",
        ".png",
        ".gif",
        ".webp",
        ".svg"
    ]

    return any(cleaned_value.endswith(extension) for extension in image_extensions)


def analyse_alt_text_rules(image: PageImage) -> dict:
    if image.alt_text is None:
        return {
            "moduleName": "AltTextRules",
            "elementType": "Image",
            "elementReference": image.src,
            "resultLabel": "Missing alt attribute",
            "severity": "High",
            "explanation": "The image does not have an alt attribute. Informative images require meaningful alternative text, while decorative images should usually use an empty alt attribute.",
            "confidenceScore": 1.0
        }

    alt_text = image.alt_text.strip()

    if alt_text == "":
        return {
            "moduleName": "AltTextRules",
            "elementType": "Image",
            "elementReference": image.src,
            "resultLabel": "Empty alt text",
            "severity": "Review",
            "explanation": "The image has empty alt text. This may be correct if the image is decorative, but should be reviewed if the image communicates useful information.",
            "confidenceScore": 0.6
        }

    cleaned_alt_text = clean_text(alt_text)

    if cleaned_alt_text in GENERIC_ALT_TEXT_VALUES:
        return {
            "moduleName": "AltTextRules",
            "elementType": "Image",
            "elementReference": image.src,
            "resultLabel": "Generic alt text",
            "severity": "Medium",
            "explanation": f"The alt text '{alt_text}' is generic and may not describe the purpose or content of the image.",
            "confidenceScore": 0.9
        }

    if looks_like_filename(alt_text):
        return {
            "moduleName": "AltTextRules",
            "elementType": "Image",
            "elementReference": image.src,
            "resultLabel": "Filename used as alt text",
            "severity": "Medium",
            "explanation": f"The alt text '{alt_text}' appears to be a file name rather than a meaningful description.",
            "confidenceScore": 0.9
        }

    if len(cleaned_alt_text.split()) == 1:
        return {
            "moduleName": "AltTextRules",
            "elementType": "Image",
            "elementReference": image.src,
            "resultLabel": "Very short alt text",
            "severity": "Low",
            "explanation": f"The alt text '{alt_text}' is very short. It may be valid in context, but should be reviewed.",
            "confidenceScore": 0.5
        }

    return {
        "moduleName": "AltTextRules",
        "elementType": "Image",
        "elementReference": image.src,
        "resultLabel": "No rule-based issue found",
        "severity": "None",
        "explanation": "No obvious rule-based alt text issue was detected.",
        "confidenceScore": 0.7
    }

In [326]:
alt_text_rule_results = [
    analyse_alt_text_rules(image)
    for image in page_images
]

alt_text_rule_results_dataframe = pd.DataFrame(alt_text_rule_results)

alt_text_rule_results_dataframe

,moduleName,elementType,elementReference,resultLabel,severity,explanation,confidenceScore
0,AltTextRules,Image,https://www.aberdeenshire.gov.uk/media/hqrc0hy...,Empty alt text,Review,The image has empty alt text. This may be corr...,0.6
1,AltTextRules,Image,https://www.aberdeenshire.gov.uk/media/rrtfeee...,Empty alt text,Review,The image has empty alt text. This may be corr...,0.6
2,AltTextRules,Image,https://www.aberdeenshire.gov.uk/media/17236/2...,Empty alt text,Review,The image has empty alt text. This may be corr...,0.6


In [327]:
alt_text_rule_findings = [
    result
    for result in alt_text_rule_results
    if result["severity"] != "None"
]

alt_text_rule_findings_dataframe = pd.DataFrame(alt_text_rule_findings)

alt_text_rule_findings_dataframe

,moduleName,elementType,elementReference,resultLabel,severity,explanation,confidenceScore
0,AltTextRules,Image,https://www.aberdeenshire.gov.uk/media/hqrc0hy...,Empty alt text,Review,The image has empty alt text. This may be corr...,0.6
1,AltTextRules,Image,https://www.aberdeenshire.gov.uk/media/rrtfeee...,Empty alt text,Review,The image has empty alt text. This may be corr...,0.6
2,AltTextRules,Image,https://www.aberdeenshire.gov.uk/media/17236/2...,Empty alt text,Review,The image has empty alt text. This may be corr...,0.6


# 12. Alt Text AI Image Captioning

This section uses an image captioning model to generate a description of each image.

The generated caption is then compared with the image's existing alt text. If the supplied alt text is very different from the generated caption, the image is flagged for review.

This does not prove that the alt text is wrong. It only identifies cases where the alt text may need human review.

In [328]:
IMAGE_CAPTIONING_MODEL_NAME = "Salesforce/blip-image-captioning-base"
TEXT_SIMILARITY_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Using device: {device}")

Using device: cpu


In [329]:
blip_processor = BlipProcessor.from_pretrained(IMAGE_CAPTIONING_MODEL_NAME)
blip_model = BlipForConditionalGeneration.from_pretrained(IMAGE_CAPTIONING_MODEL_NAME)

blip_model.to(device)

text_similarity_model = SentenceTransformer(TEXT_SIMILARITY_MODEL_NAME)

The first time this cell runs, the models may need to be downloaded. Subsequent runs should be faster if the models are cached locally.

In [330]:
def load_image_from_url(image_url: str) -> Optional[Image.Image]:
    try:
        response = requests.get(
            image_url,
            timeout=REQUEST_TIMEOUT_SECONDS
        )

        response.raise_for_status()

        image = Image.open(io.BytesIO(response.content))
        image = image.convert("RGB")

        return image

    except Exception as exception:
        print(f"Could not load image: {image_url}")
        print(f"Reason: {exception}")

        return None

In [331]:
def generate_image_caption(image: Image.Image) -> str:
    inputs = blip_processor(
        image,
        return_tensors="pt"
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    output = blip_model.generate(
        **inputs,
        max_new_tokens=30
    )

    caption = blip_processor.decode(
        output[0],
        skip_special_tokens=True
    )

    return caption

In [332]:
def calculate_semantic_similarity(first_text: str, second_text: str) -> float:
    first_embedding = text_similarity_model.encode(
        first_text,
        convert_to_tensor=True
    )

    second_embedding = text_similarity_model.encode(
        second_text,
        convert_to_tensor=True
    )

    similarity_score = util.cos_sim(
        first_embedding,
        second_embedding
    )

    return float(similarity_score[0][0])

In [ ]:
def analyse_alt_text_with_ai(image: PageImage) -> dict:
    if image.alt_text is None:
        return {
            "moduleName": "AltTextAI",
            "elementType": "Image",
            "elementReference": image.src,
            "resultLabel": "Skipped",
            "severity": "None",
            "explanation": "AI comparison was skipped because the image does not have an alt attribute.",
            "confidenceScore": 0.0
        }

    alt_text = image.alt_text.strip()

    if alt_text == "":
        return {
            "moduleName": "AltTextAI",
            "elementType": "Image",
            "elementReference": image.src,
            "resultLabel": "Skipped",
            "severity": "None",
            "explanation": "AI comparison was skipped because the image has empty alt text. This may be correct if the image is decorative.",
            "confidenceScore": 0.0
        }

    loaded_image = load_image_from_url(image.src)

    if loaded_image is None:
        return {
            "moduleName": "AltTextAI",
            "elementType": "Image",
            "elementReference": image.src,
            "resultLabel": "Image could not be loaded",
            "severity": "Review",
            "explanation": "The image could not be loaded for AI captioning.",
            "confidenceScore": 0.0
        }

    generated_caption = generate_image_caption(loaded_image)

    similarity_score = calculate_semantic_similarity(
        alt_text,
        generated_caption
    )

    if similarity_score < 0.30:
        result_label = "Needs human review"
        severity = "Low"
        explanation = f"The supplied alt text '{alt_text}' has low similarity to the AI-generated caption '{generated_caption}'. This may indicate a mismatch, but the result should be reviewed by a human because the image captioning model may be incorrect."
    elif similarity_score < 0.50:
        result_label = "Needs review"
        severity = "Low"
        explanation = f"The supplied alt text '{alt_text}' has moderate similarity to the AI-generated caption '{generated_caption}'."
    else:
        result_label = "Likely relevant"
        severity = "None"
        explanation = f"The supplied alt text '{alt_text}' appears broadly similar to the AI-generated caption '{generated_caption}'."

    return {
        "moduleName": "AltTextAI",
        "elementType": "Image",
        "elementReference": image.src,
        "resultLabel": result_label,
        "severity": severity,
        "explanation": explanation,
        "confidenceScore": round(similarity_score, 3)
    }

In [350]:
images_for_ai_analysis = [
    image
    for image, rule_result in zip(page_images, alt_text_rule_results)
    if rule_result["severity"] == "None"
]

print(f"Images selected for AI captioning: {len(images_for_ai_analysis)}")

alt_text_ai_results = [
    analyse_alt_text_with_ai(image)
    for image in images_for_ai_analysis
]

alt_text_ai_results_dataframe = pd.DataFrame(alt_text_ai_results)

alt_text_ai_results_dataframe

Images selected for AI captioning: 0


""


In [335]:
alt_text_ai_findings = [
    result
    for result in alt_text_ai_results
    if result["severity"] not in ["None"]
]

alt_text_ai_findings_dataframe = pd.DataFrame(alt_text_ai_findings)

display(alt_text_ai_findings_dataframe)

""


In [336]:
alt_text_ai_findings = [
    result
    for result in alt_text_ai_results
    if result["severity"] not in ["None"]
]

alt_text_ai_findings_dataframe = pd.DataFrame(alt_text_ai_findings)

alt_text_ai_findings_dataframe

""


# 13. Heading Meaningfulness Analysis

This section uses simple NLP techniques to estimate whether each heading is meaningful.

The analysis checks whether headings are:

- Generic
- Very short
- Missing related content
- Weakly related to the content that follows them

The results should be treated as indicators for review rather than definitive accessibility failures.

In [ ]:
GENERIC_HEADING_VALUES = {
    "more",
    "information",
    "details",
    "useful links",
    "links",
    "read more",
    "learn more",
    "content",
    "section",
    "page section",
    "other",
    "miscellaneous",
    "services",
    "help",
    "overview"
}

In [338]:
def get_heading_confidence_from_similarity(similarity_score: float) -> float:
    if similarity_score < 0.15:
        return 0.6

    if similarity_score < 0.25:
        return 0.5

    return 0.4

In [351]:
def get_heading_confidence_from_similarity(similarity_score: float) -> float:
    if similarity_score < 0.15:
        return 0.6

    if similarity_score < 0.25:
        return 0.5

    return 0.4

In [ ]:
def analyse_heading_meaningfulness(heading: PageHeading) -> dict:
    heading_text = heading.text.strip()
    cleaned_heading_text = clean_text(heading_text)
    element_reference = f"h{heading.level}: {heading_text}"

    if cleaned_heading_text in GENERIC_HEADING_VALUES:
        return {
            "moduleName": "HeadingMeaningfulness",
            "elementType": "Heading",
            "elementReference": element_reference,
            "resultLabel": "Generic heading",
            "severity": "Medium",
            "explanation": f"The heading '{heading_text}' appears generic and may not clearly describe the topic or purpose of the following content.",
            "confidenceScore": 0.85
        }

    if heading.following_text.strip() == "":
        return {
            "moduleName": "HeadingMeaningfulness",
            "elementType": "Heading",
            "elementReference": element_reference,
            "resultLabel": "No related content found",
            "severity": "Review",
            "explanation": f"The heading '{heading_text}' does not appear to have related content directly following it in the parsed HTML.",
            "confidenceScore": 0.5
        }

    if len(cleaned_heading_text.split()) == 1:
        return {
            "moduleName": "HeadingMeaningfulness",
            "elementType": "Heading",
            "elementReference": element_reference,
            "resultLabel": "Very short heading",
            "severity": "Low",
            "explanation": f"The heading '{heading_text}' is very short. It may be meaningful in context, but should be reviewed.",
            "confidenceScore": 0.55
        }

    similarity_score = calculate_semantic_similarity(
        heading.text,
        heading.following_text
    )

    if similarity_score < 0.25:
        confidence_score = get_heading_confidence_from_similarity(similarity_score)

        return {
            "moduleName": "HeadingMeaningfulness",
            "elementType": "Heading",
            "elementReference": element_reference,
            "resultLabel": "Needs review",
            "severity": "Low",
            "explanation": f"The heading '{heading_text}' has low semantic similarity to the extracted following content. Similarity score: {round(similarity_score, 3)}. This may indicate a weak heading, but should be reviewed because the extracted content may not fully represent the page section.",
            "confidenceScore": confidence_score
        }

    return {
        "moduleName": "HeadingMeaningfulness",
        "elementType": "Heading",
        "elementReference": element_reference,
        "resultLabel": "Likely meaningful",
        "severity": "None",
        "explanation": f"The heading '{heading_text}' appears to be semantically related to the following content. Similarity score: {round(similarity_score, 3)}.",
        "confidenceScore": 0.7
    }

In [340]:
heading_results = [
    analyse_heading_meaningfulness(heading)
    for heading in page_headings
]

heading_results_dataframe = pd.DataFrame(heading_results)

heading_results_dataframe

,moduleName,elementType,elementReference,resultLabel,severity,explanation,confidenceScore
0,HeadingMeaningfulness,Heading,h1: Housing and community buildings,Needs review,Low,The heading 'Housing and community buildings' ...,0.6
1,HeadingMeaningfulness,Heading,h2: Affordable housing development,Likely meaningful,None,The heading 'Affordable housing development' a...,0.7
2,HeadingMeaningfulness,Heading,h2: New council building progress,Likely meaningful,None,The heading 'New council building progress' ap...,0.7
3,HeadingMeaningfulness,Heading,h2: Decorative separator image,Likely meaningful,None,The heading 'Decorative separator image' appea...,0.7
4,HeadingMeaningfulness,Heading,h2: Review required,Needs review,Low,The heading 'Review required' has low semantic...,0.6


In [341]:
heading_findings = [
    result
    for result in heading_results
    if result["severity"] != "None"
]

heading_findings_dataframe = pd.DataFrame(heading_findings)

heading_findings_dataframe

,moduleName,elementType,elementReference,resultLabel,severity,explanation,confidenceScore
0,HeadingMeaningfulness,Heading,h1: Housing and community buildings,Needs review,Low,The heading 'Housing and community buildings' ...,0.6
1,HeadingMeaningfulness,Heading,h2: Review required,Needs review,Low,The heading 'Review required' has low semantic...,0.6


# 14. Combine AI Findings

This section combines the findings from the alt text and heading analysis modules.

Only findings with a severity other than `None` are prepared for submission to the API.

In [342]:
all_ai_findings = []

all_ai_findings.extend(alt_text_rule_findings)
all_ai_findings.extend(alt_text_ai_findings)
all_ai_findings.extend(heading_findings)

print(f"Total AI findings to submit: {len(all_ai_findings)}")

Total AI findings to submit: 5


In [343]:
all_ai_findings_dataframe = pd.DataFrame(all_ai_findings)

all_ai_findings_dataframe

,moduleName,elementType,elementReference,resultLabel,severity,explanation,confidenceScore
0,AltTextRules,Image,https://www.aberdeenshire.gov.uk/media/hqrc0hy...,Empty alt text,Review,The image has empty alt text. This may be corr...,0.6
1,AltTextRules,Image,https://www.aberdeenshire.gov.uk/media/rrtfeee...,Empty alt text,Review,The image has empty alt text. This may be corr...,0.6
2,AltTextRules,Image,https://www.aberdeenshire.gov.uk/media/17236/2...,Empty alt text,Review,The image has empty alt text. This may be corr...,0.6
3,HeadingMeaningfulness,Heading,h1: Housing and community buildings,Needs review,Low,The heading 'Housing and community buildings' ...,0.6
4,HeadingMeaningfulness,Heading,h2: Review required,Needs review,Low,The heading 'Review required' has low semantic...,0.6


The combined findings table gives a final overview of the issues identified by the proof-of-concept AI modules before they are submitted to the API.

# 15. Submit AI Findings to the API

This section submits each AI finding to the project API.

Each finding is posted against the selected scan request using the scan ID.

In [344]:
for finding in all_ai_findings:
    post_ai_finding(
        scan_id=scan_id,
        finding=finding
    )

print(f"Submitted {len(all_ai_findings)} AI findings for scan ID {scan_id}.")

C:\Users\cfowler\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'localhost'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
C:\Users\cfowler\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'localhost'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
C:\Users\cfowler\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\urllib3\connectionpool.py:1097: InsecureRequest

Submitted 5 AI findings for scan ID 31.


# 16. Mark AI Scan as Complete

This section marks the selected scan request as complete after the AI findings have been reviewed or submitted.

In [345]:
MARK_SCAN_AS_COMPLETE = True

if MARK_SCAN_AS_COMPLETE:
    mark_ai_scan_complete(scan_id)

    print(f"AI scan marked as complete for scan ID {scan_id}.")
else:
    print("Scan completion skipped. Set MARK_SCAN_AS_COMPLETE to True to update the API.")

AI scan marked as complete for scan ID 31.


C:\Users\cfowler\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'localhost'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


# 17. Display Final Notebook Results

This section displays a final summary of the AI analysis results.

These tables can be used as evidence in the project appendix and may also be useful for screenshots in the main report.

In [346]:
summary_data = {
    "Scan ID": [scan_id],
    "Scan URL": [scan_url],
    "Images found": [len(page_images)],
    "Headings found": [len(page_headings)],
    "Alt text rule findings": [len(alt_text_rule_findings)],
    "Alt text AI findings": [len(alt_text_ai_findings)],
    "Heading findings": [len(heading_findings)],
    "Total AI findings": [len(all_ai_findings)]
}

summary_dataframe = pd.DataFrame(summary_data)

summary_dataframe

,Scan ID,Scan URL,Images found,Headings found,Alt text rule findings,Alt text AI findings,Heading findings,Total AI findings
0,31,http://localhost:8000/mock_pages/05_severe_com...,3,5,3,0,2,5


In [347]:
if len(all_ai_findings) > 0:
    final_findings_dataframe = pd.DataFrame(all_ai_findings)

    display(final_findings_dataframe)
else:
    print("No AI findings were identified for this scan.")

,moduleName,elementType,elementReference,resultLabel,severity,explanation,confidenceScore
0,AltTextRules,Image,https://www.aberdeenshire.gov.uk/media/hqrc0hy...,Empty alt text,Review,The image has empty alt text. This may be corr...,0.6
1,AltTextRules,Image,https://www.aberdeenshire.gov.uk/media/rrtfeee...,Empty alt text,Review,The image has empty alt text. This may be corr...,0.6
2,AltTextRules,Image,https://www.aberdeenshire.gov.uk/media/17236/2...,Empty alt text,Review,The image has empty alt text. This may be corr...,0.6
3,HeadingMeaningfulness,Heading,h1: Housing and community buildings,Needs review,Low,The heading 'Housing and community buildings' ...,0.6
4,HeadingMeaningfulness,Heading,h2: Review required,Needs review,Low,The heading 'Review required' has low semantic...,0.6


In [348]:
if len(all_ai_findings) > 0:
    findings_by_module_dataframe = (
        pd.DataFrame(all_ai_findings)
        .groupby(["moduleName", "severity"])
        .size()
        .reset_index(name="count")
    )

    display(findings_by_module_dataframe)
else:
    print("No findings to group.")

,moduleName,severity,count
0,AltTextRules,Review,3
1,HeadingMeaningfulness,Low,2


The summary table provides a simple overview of the scan and the number of findings identified by each module.